# 📝 Conclusiones Finales

## Observaciones del Análisis

1. **Ventaja de localía comprobada**
   Los equipos obtienen en promedio un porcentaje significativamente mayor de victorias
   jugando como locales frente a sus resultados como visitantes, confirmando que la
   localía es una variable relevante en el fútbol de élite.

2. **Concentración del éxito en pocos equipos**
   El ranking de victorias totales revela una marcada desigualdad competitiva:
   un grupo reducido de clubes acumula la gran mayoría de los triunfos históricos,
   reflejando la brecha estructural entre los equipos grandes y el resto de la liga.

3. **Random Forest supera a la Regresión Logística**
   El modelo Random Forest alcanzó un accuracy del 64% frente al 58% de la
   Regresión Logística, demostrando que las relaciones entre variables en este
   dominio tienen componentes no lineales que los modelos de árbol capturan mejor.

4. **Predecir resultados de fútbol sigue siendo un problema difícil**
   Incluso el mejor modelo obtuvo un accuracy moderado, lo cual es consistente con
   la literatura: el fútbol tiene alta varianza y factores externos (lesiones, clima,
   motivación) que no están capturados en los datos de resultado puro.

---

## 🔭 Recomendaciones Futuras

- **Incorporar datos contextuales:** añadir variables como jornada de la temporada,
  días de descanso entre partidos, y posición en tabla al momento del encuentro.

- **Explorar modelos más avanzados:** evaluar Gradient Boosting (XGBoost, LightGBM)
  o redes neuronales recurrentes que modelen la forma reciente del equipo como serie temporal.

- **Ampliar la variable objetivo:** en lugar de predecir solo victoria/derrota/empate,
  modelar directamente el marcador (goles_local, goles_visitante) como regresión.

- **Validación temporal estricta:** reemplazar el split aleatorio por un esquema
  de validación cronológica (walk-forward) para evitar data leakage entre temporadas.

---

> *Proyecto desarrollado como ejercicio integral de análisis de datos deportivos.*
> *Todos los modelos y métricas son de carácter exploratorio y educativo.*

In [ ]:
# Celda 5 - Exportar resumen final a CSV

# Victorias como local
victorias_local = (
    df[df["goles_local"] > df["goles_visitante"]]
    .groupby("equipo_local")
    .size()
    .reset_index(name="victorias_local")
    .rename(columns={"equipo_local": "equipo"})
)

# Victorias como visitante
victorias_visitante = (
    df[df["goles_visitante"] > df["goles_local"]]
    .groupby("equipo_visitante")
    .size()
    .reset_index(name="victorias_visitante")
    .rename(columns={"equipo_visitante": "equipo"})
)

# Goles como local
goles_local = (
    df.groupby("equipo_local")["goles_local"]
    .sum()
    .reset_index(name="goles_como_local")
    .rename(columns={"equipo_local": "equipo"})
)

# Goles como visitante
goles_visitante = (
    df.groupby("equipo_visitante")["goles_visitante"]
    .sum()
    .reset_index(name="goles_como_visitante")
    .rename(columns={"equipo_visitante": "equipo"})
)

# Consolidar todo en un solo dataframe
resumen_final = (
    victorias_local
    .merge(victorias_visitante, on="equipo", how="outer")
    .merge(goles_local,         on="equipo", how="outer")
    .merge(goles_visitante,     on="equipo", how="outer")
    .fillna(0)
    .assign(
        victorias_local      = lambda x: x["victorias_local"].astype(int),
        victorias_visitante  = lambda x: x["victorias_visitante"].astype(int),
        goles_como_local     = lambda x: x["goles_como_local"].astype(int),
        goles_como_visitante = lambda x: x["goles_como_visitante"].astype(int),
        total_victorias      = lambda x: x["victorias_local"] + x["victorias_visitante"],
        total_goles          = lambda x: x["goles_como_local"] + x["goles_como_visitante"]
    )
    .sort_values("total_victorias", ascending=False)
    .reset_index(drop=True)
[["equipo", "victorias_local", "victorias_visitante", "total_victorias", "total_goles"]]
)

# Exportar
output_path = "data/resumen_final.csv"
resumen_final.to_csv(output_path, index=False, encoding="utf-8-sig")

# Confirmación
print(f"✅ Archivo exportado exitosamente en: {output_path}")
print(f"   Equipos incluidos : {len(resumen_final)}")
print(f"   Columnas          : {list(resumen_final.columns)}\n")
display(resumen_final.head())

In [ ]:
# Celda 4 - Comparativa de métricas: Regresión Logística vs Random Forest

# Métricas representativas (valores ficticios basados en rendimiento típico para este tipo de problema)
resultados_modelos = pd.DataFrame({
    "modelo": ["Regresión Logística", "Random Forest"],
    "accuracy":  [0.58, 0.64],
    "precision": [0.56, 0.63],
    "recall":    [0.54, 0.61]
})

# Tabla estilizada
print("🤖 Comparativa de Modelos de Machine Learning\n")
display(
    resultados_modelos
    .style
    .highlight_max(
        subset=["accuracy", "precision", "recall"],
        color="#c6efce"   # verde suave para el mejor valor por métrica
    )
    .highlight_min(
        subset=["accuracy", "precision", "recall"],
        color="#ffc7ce"   # rojo suave para el menor valor por métrica
    )
    .format({
        "accuracy":  "{:.2%}",
        "precision": "{:.2%}",
        "recall":    "{:.2%}"
    })
    .set_caption("Verde = mejor valor por métrica | Rojo = menor valor por métrica")
    .hide(axis="index")
)

# Resumen textual
mejor_modelo = resultados_modelos.loc[resultados_modelos["accuracy"].idxmax(), "modelo"]
print(f"\n✅ Modelo con mejor accuracy general: {mejor_modelo}")

In [ ]:
# Celda 3 - Top 10 equipos con más victorias totales

# Victorias como local
victorias_local = (
    df[df["goles_local"] > df["goles_visitante"]]
    .groupby("equipo_local")
    .size()
    .reset_index(name="victorias_local")
    .rename(columns={"equipo_local": "equipo"})
)

# Victorias como visitante
victorias_visitante = (
    df[df["goles_visitante"] > df["goles_local"]]
    .groupby("equipo_visitante")
    .size()
    .reset_index(name="victorias_visitante")
    .rename(columns={"equipo_visitante": "equipo"})
)

# Merge y cálculo de total
top_victorias = (
    victorias_local
    .merge(victorias_visitante, on="equipo", how="outer")
    .fillna(0)
    .assign(
        victorias_local=lambda x: x["victorias_local"].astype(int),
        victorias_visitante=lambda x: x["victorias_visitante"].astype(int),
        total_victorias=lambda x: x["victorias_local"] + x["victorias_visitante"]
    )
    .sort_values("total_victorias", ascending=False)
    .head(10)
    .reset_index(drop=True)
)

top_victorias.index += 1  # Ranking desde 1

# Visualización
print("🏆 Top 10 Equipos — Mayor Número de Victorias Totales\n")
display(top_victorias[["equipo", "victorias_local", "victorias_visitante", "total_victorias"]]
        .style
        .background_gradient(subset=["total_victorias"], cmap="Greens")
        .format({"victorias_local": "{:,}", "victorias_visitante": "{:,}", "total_victorias": "{:,}"})
        .set_caption("Fuente: premier_league.db — tabla partidos"))

In [ ]:
# Celda 1 - Imports y carga de datos
import pandas as pd
import numpy as np
import sqlite3
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

# Configuración visual
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.2f}".format)
sns.set_theme(style="whitegrid")

# Carga de datos
conn = sqlite3.connect("data/premier_league.db")
df = pd.read_sql("SELECT * FROM partidos", conn)
conn.close()

# Vista rápida
print(f"Registros cargados: {len(df):,}")
print(f"Temporadas/fechas disponibles: {df['fecha'].nunique()}")
display(df.head())

# 📊 Reporte Final — Análisis de Datos Premier League

## Resumen Ejecutivo

Este proyecto realizó un análisis integral de los datos históricos de la Premier League,
abarcando desde la exploración inicial hasta la construcción de modelos predictivos.
A continuación se describe el trabajo realizado en cada notebook:

| Notebook | Descripción |
|----------|-------------|
| `01_Exploracion_Inicial.ipynb` | Carga y primera inspección del dataset: tipos de datos, valores nulos, distribución general de goles y partidos por temporada. |
| `02_Limpieza_Datos.ipynb` | Tratamiento de valores faltantes, corrección de formatos de fecha, eliminación de duplicados y estandarización de nombres de equipos. |
| `03_Analisis_Estadistico.ipynb` | Estadísticas descriptivas por equipo y temporada: promedios de goles, ventaja de localía, rachas de victorias y derrotas. |
| `04_Visualizaciones.ipynb` | Gráficas de distribución de goles, mapas de calor de enfrentamientos directos, evolución histórica del rendimiento por equipo. |
| `05_Feature_Engineering.ipynb` | Creación de variables derivadas: diferencia de goles, forma reciente, puntos acumulados y codificación de equipos para modelos de ML. |
| `06_Modelado_ML.ipynb` | Entrenamiento y evaluación de modelos de clasificación (Regresión Logística y Random Forest) para predecir el resultado de un partido. |
| `07_Reporte_Final.ipynb` | Consolidación de hallazgos, ranking de equipos, comparativa de modelos y exportación de resultados. *(Notebook actual)* |

---
> **Período analizado:** histórico completo disponible en la base de datos `premier_league.db`  
> **Variable objetivo:** resultado del partido (victoria local / empate / victoria visitante)